# VisionMirror CatVTON Google Colab GPU Server
This notebook installs CatVTON, starts a native FastAPI model server on GPU, tunnels it via Cloudflare Tunnel, and prints the public API URL.

In [ ]:
# 1. Verify GPU availability
import torch
assert torch.cuda.is_available(), "GPU is not enabled! Please go to Runtime -> Change runtime type and select GPU (T4/A100)."

In [ ]:
# 2. Clone repository & install dependencies
%cd /content
!rm -rf VisionMirror
!git clone https://github.com/swarnaverma10/VisionMirror.git
%cd /content/VisionMirror/catvton

# Install standard packages
!pip install -r requirements.txt
!pip install fastapi uvicorn python-multipart

# Robustly install detectron2 and official DensePose module
import torch
import sys
import subprocess
import os

torch_ver = ".".join(torch.__version__.split(".")[:2])
if torch.cuda.is_available() and "+" in torch.__version__:
    cuda_ver = torch.__version__.split("+")[-1]
else:
    cuda_ver = "cpu"

print(f"Detected PyTorch: {torch_ver}, CUDA: {cuda_ver}")

# Step A: Install Detectron2
print("Installing detectron2...")
installed = False

# Try MiroPsota package builder wheels index first
try:
    subprocess.run([
        sys.executable, "-m", "pip", "install", 
        "--extra-index-url", "https://miropsota.github.io/torch_packages_builder",
        "detectron2"
    ], check=True)
    installed = True
    print("Successfully installed detectron2 from community wheel index.")
except Exception as e:
    print(f"Community wheel install failed: {e}")

if not installed:
    # Try official FAIR wheels index
    wheel_url = f"https://dl.fbaipublicfiles.com/detectron2/wheels/{cuda_ver}/torch{torch_ver}/index.html"
    try:
        subprocess.run([
            sys.executable, "-m", "pip", "install", "detectron2",
            "-f", wheel_url
        ], check=True)
        installed = True
        print("Successfully installed detectron2 from official FAIR wheels index.")
    except Exception as e:
        print(f"FAIR wheel install failed: {e}")

if not installed:
    # Fallback to source compilation with proper environment variables
    print("Compiling detectron2 from source (this might take a few minutes)...")
    env = os.environ.copy()
    env["FORCE_CUDA"] = "1"
    env["TORCH_CUDA_ARCH_LIST"] = "8.0;8.6;8.9;9.0"
    try:
        subprocess.run([
            sys.executable, "-m", "pip", "install", 
            "git+https://github.com/facebookresearch/detectron2.git"
        ], env=env, check=True)
        installed = True
        print("Successfully compiled and installed detectron2 from source.")
    except Exception as e:
        print(f"Compilation from source failed: {e}")
        sys.exit(1)

# Step B: Install DensePose from the official detectron2 repository subdirectory
print("Installing DensePose from official detectron2 projects/DensePose...")
try:
    subprocess.run([
        sys.executable, "-m", "pip", "install", 
        "git+https://github.com/facebookresearch/detectron2.git@v0.6#subdirectory=projects/DensePose"
    ], check=True)
    print("DensePose installed successfully!")
except Exception as e:
    print(f"Failed to install DensePose from official repo: {e}")
    sys.exit(1)

# Step C: Verification of the three target commands
print("Running verification tests...")

print("\n--- Verification 1: python -c \"import densepose\" ---")
res1 = subprocess.run([sys.executable, "-c", "import densepose; print(densepose.__file__)"], capture_output=True, text=True)
print(f"STDOUT: {res1.stdout}")
print(f"STDERR: {res1.stderr}")
assert res1.returncode == 0, "Verification 1 failed!"

print("--- Verification 2: python -c \"from model.DensePose import DensePose\" ---")
res2 = subprocess.run([sys.executable, "-c", "from model.DensePose import DensePose; print('DensePose wrapper loaded successfully!')"], capture_output=True, text=True)
print(f"STDOUT: {res2.stdout}")
print(f"STDERR: {res2.stderr}")
assert res2.returncode == 0, "Verification 2 failed!"

print("--- Verification 3: python -c \"from colab_server import app\" ---")
res3 = subprocess.run([sys.executable, "-c", "from colab_server import app; print('colab_server loaded successfully!')"], capture_output=True, text=True)
print(f"STDOUT: {res3.stdout}")
print(f"STDERR: {res3.stderr}")
assert res3.returncode == 0, "Verification 3 failed!"

print("All verification checks passed successfully!")

In [ ]:
# 3. Run FastAPI and Cloudflare Tunnel
import subprocess
import time
import re
import sys
import urllib.request
import urllib.error

# Verify current directory state
print("--- PATH & FILE VERIFICATION ---")
subprocess.run(["pwd"])
subprocess.run(["ls"])
subprocess.run(["find", ".", "-name", "colab_server.py"])
print("--------------------------------\n")

# Download Cloudflare Tunnel client
print("Downloading cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start FastAPI server in the background (explicitly forcing cwd to /content/VisionMirror/catvton)
print("Starting FastAPI server...")
print("Command: uvicorn colab_server:app --host 127.0.0.1 --port 8000 --app-dir /content/VisionMirror/catvton")
server_proc = subprocess.Popen(
    ["uvicorn", "colab_server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/VisionMirror/catvton",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for FastAPI server to start
print("Waiting for FastAPI server to load models and bind to port 8000...")
started = False
for i in range(120):
    if server_proc.poll() is not None:
        print("FastAPI server process crashed on startup!")
        out = server_proc.stdout.read()
        print(out)
        sys.exit(1)
    try:
        response = urllib.request.urlopen("http://127.0.0.1:8000/", timeout=1)
        if response.status == 200:
            print("FastAPI server is up and healthy!")
            started = True
            break
    except Exception:
        pass
    time.sleep(1)

if not started:
    print("FastAPI server failed to start within 120 seconds.")
    server_proc.terminate()
    sys.exit(1)

# Start Cloudflare Tunnel in the background
print("Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(5)  # Wait for tunnel initialization

# Listen to tunnel output and extract the public URL
public_url = None
try:
    for line in iter(tunnel_proc.stdout.readline, ""):
        sys.stdout.write(line)
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "="*50)
            print(f"PUBLIC API URL: {public_url}")
            print("="*50 + "\n")
            break
except KeyboardInterrupt:
    print("Stopping...")
    server_proc.terminate()
    tunnel_proc.terminate()

# Keep running to stream logs
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            sys.stdout.write(line)
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopping server...")
    server_proc.terminate()
    tunnel_proc.terminate()